### `etd_tests.ipynb` 
*Created: Sept 22, 2026* <br/>
Notebook for testing some Exponential Time Differencing (ETD) solvers by comparing them against trusted reference solvers from `OrdinaryDiffEq.jl`. We also test against exact solutions, when available.

In [16]:
using OrdinaryDiffEq, CairoMakie, NBInclude, UnPack, Printf, Test, LinearAlgebra, LaTeXStrings, Statistics
using OrdinaryDiffEqExponentialRK, SciMLOperators 
import OrdinaryDiffEqCore: OrdinaryDiffEqAlgorithm  

In [15]:
#Import the ETD solvers 
@nbinclude("etd_euler.ipynb")
@nbinclude("etd_rk2.ipynb")
@nbinclude("etd_rk3.ipynb")
@nbinclude("etd_rk4.ipynb")

#Import ODE test problems
@nbinclude("tests/test_problems.ipynb")

#Use desired plotting defaults for Makie 
@nbinclude("../../../../../../set_makie_defaults.ipynb")

In [17]:
struct SemilinearTestProblem{M, F, U, P, S}
    A::M
    f::F
    u0::U
    tspan::Tuple{Float64, Float64}
    p::P
    exact_solution::S
end 

function SemilinearTestProblem(A::M, f::F, u0::U, tspan::NTuple{2,<:Real}, p::P = nothing; exact_solution::S = nothing) where {M,F,U,P,S}
    tspan = Float64.(tspan)
    return SemilinearTestProblem(A, f, u0, tspan, p, exact_solution)
end

SemilinearTestProblem

In [37]:
function etd_euler(prob::SemilinearTestProblem; Δt::Float64, ϵ::Float64 = 1e-3)
    return etd_euler(prob.A, prob.f, prob.u0, prob.tspan, prob.p; Δt = Δt, ϵ = ϵ)
end 

etd_euler (generic function with 3 methods)

#### **Test 1: A Scalar Riccati Equation**

##### **ODE: $\displaystyle u' = -au + u^2, \quad u(0) = u_0, \quad 0 < u_0 < a$**

##### **Exact Solution:**  $\quad \displaystyle u(t) = \frac{a}{1 - \left(1 - \frac{a}{u_0} \right)e^{at}}$

- We have $A = -a$ and $f(u,t) = u^2$
- Note that $\lim\limits_{t \to \infty} u(t) = 0$, provided $a > 0$. 

In [38]:
#Test ETD Euler using a *scalar* equation
a = 3.0
A = -a 
u0 = 1.0 
p = (a = a, u0 = u0)
riccati_rhs(u,p,t) = u^2 

function riccati_exact(t,p)
    @unpack a, u0 = p
    return a ./ (1.0 - (1.0 - a/u0)*exp(a*t))
end 

riccati = SemilinearTestProblem(A, riccati_rhs, u0, (0.0, 2.0), p; exact_solution = riccati_exact);

#Now solve the riccati equation using `etd_euler`
sol = etd_euler(riccati; Δt = 0.01)

u_approx = sol.u
u_exact = riccati_exact.(sol.t, Ref(sol.p));

l_inf_norm = maximum(u_approx .- u_exact)
@printf("l_inf_error = %.4e", l_inf_norm)

# fig = Figure(size = (400,400))
# ax = Axis(fig[1,1], xlabel = L"t", ylabel = L"u", title = "ETD Euler")
# scatter!(sol.t, u_approx, label = "numerical", color = :orange, markersize = 6)
# lines!(sol.t, u_exact, label = "exact", color = :blue)
# axislegend(position = :rt, ax)
# display(fig)

l_inf_error = 2.8202e-03

#### **Test 2: Nonlinear System** 
$$A = \begin{pmatrix} -100 & \phantom{-}0 \\ 0 & -1 \end{pmatrix}, \quad F(\mathbf{u},p,t) = \begin{pmatrix} \cos t + 100 \sin t + u_1^2 - \sin^2 t \\ - \sin t + \cos t + u_1 u_2 - \sin t \cos t \end{pmatrix}, \quad \mathbf{u}(0) = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

##### **Exact solution:** $$\mathbf{u}(t) = \begin{pmatrix} u_1(t) \\ u_2(t) \end{pmatrix} = \begin{pmatrix} \sin t \\ \cos t \end{pmatrix}$$



In [47]:
function rotation_rhs(u,p,t)
    f1 = cos(t) + 100*sin(t) + (u[1])^2 - (sin(t))^2 
    f2 = -sin(t) + cos(t) + u[1] * u[2] - sin(t) * cos(t)
    return [f1, f2]
end 

rotation_exact(t, p) = [sin(t), cos(t)]

A = [-100.0 0.0; 0.0 -1.0]
u0 = [0.0, 1.0]
p = nothing

#Now solve the rotation equation using `etd_euler`
rotation = SemilinearTestProblem(A, rotation_rhs, u0, (0.0, 2.0), p; exact_solution = rotation_exact);
sol = etd_euler(rotation; Δt = 0.01);

u_approx = sol.u
u_exact = rotation_exact.(sol.t, Ref(sol.p));

l_inf_norm = maximum(map(err -> norm(err), u_approx .- u_exact))
@printf("l_inf_error = %.4e", l_inf_norm)

l_inf_error = 7.0043e-03